# EXP-001 — χ Discrimination Benchmark

Tests whether QRR's collapse index χ discriminates semantically ambiguous inputs from clear ones, **without any fine-tuning**.

**Hypothesis**: Mean χ(ambiguous) − Mean χ(clear) > 0.15  
**Success criterion**: Δχ > 0.15, p < 0.05 (Mann-Whitney U)

In [ ]:
# Install dependencies (run once)
# !pip install -e ..[dev,benchmarks]

In [ ]:
import sys, torch
sys.path.insert(0, '..')
from benchmarks.ambiguity_bench import run_exp001, AMBIGUOUS_INPUTS, CLEAR_INPUTS
from qrr.qrr_model import QRRModel
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')

## 1. Load Model

In [ ]:
model = QRRModel(base_model_name='gpt2', k_branches=4, freeze_base=True)
model.eval()
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'QRR trainable parameters: {n_params:,}')

## 2. Run EXP-001 (full benchmark)

In [ ]:
results = run_exp001(
    model_name='gpt2',
    k_branches=4,
    n_samples=30,
    output_path='../experiments/EXP-001_results.json',
    verbose=True,
)

## 3. Visualize Results

In [ ]:
# Recompute per-sample χ for plotting
chi_a, chi_c = [], []
with torch.no_grad():
    for t in AMBIGUOUS_INPUTS[:30]:
        chi_a.append(model.compute_chi(t))
    for t in CLEAR_INPUTS[:30]:
        chi_c.append(model.compute_chi(t))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution plot
axes[0].hist(chi_a, bins=15, alpha=0.7, color='#e74c3c', label='Ambiguous')
axes[0].hist(chi_c, bins=15, alpha=0.7, color='#2ecc71', label='Clear')
axes[0].axvline(np.mean(chi_a), color='#c0392b', linestyle='--', label=f'Mean ambiguous: {np.mean(chi_a):.3f}')
axes[0].axvline(np.mean(chi_c), color='#27ae60', linestyle='--', label=f'Mean clear: {np.mean(chi_c):.3f}')
axes[0].set_xlabel('χ (Collapse Index)')
axes[0].set_ylabel('Count')
axes[0].set_title('EXP-001: χ Distribution')
axes[0].legend()

# Scatter plot
axes[1].scatter(range(len(chi_a)), chi_a, c='#e74c3c', alpha=0.7, label='Ambiguous', s=40)
axes[1].scatter(range(len(chi_c)), chi_c, c='#2ecc71', alpha=0.7, label='Clear', s=40)
axes[1].axhline(0.15, color='gray', linestyle=':', label='χ=0.15')
axes[1].set_xlabel('Sample index')
axes[1].set_ylabel('χ')
axes[1].set_title('Per-sample χ scores')
axes[1].legend()

plt.tight_layout()
plt.savefig('../experiments/EXP-001_chi_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Δχ = {np.mean(chi_a) - np.mean(chi_c):.4f}')

## 4. Ablation — Effect of K

In [ ]:
# Test K = 2, 4, 8 to see impact on Δχ
k_values = [2, 4, 8]
delta_chis = []
for k in k_values:
    r = run_exp001(model_name='gpt2', k_branches=k, n_samples=20, verbose=False)
    delta_chis.append(r['delta_chi'])
    print(f'K={k}: Δχ={r["delta_chi"]:.4f}, p={r["p_value"]:.4f}')

plt.figure(figsize=(6, 4))
plt.plot(k_values, delta_chis, 'o-', color='#2980b9')
plt.axhline(0.15, color='red', linestyle='--', label='Success threshold')
plt.xlabel('K (branches)')
plt.ylabel('Δχ')
plt.title('Δχ vs K — EXP-001 Ablation')
plt.legend()
plt.savefig('../experiments/EXP-001_k_ablation.png', dpi=150, bbox_inches='tight')
plt.show()